# [1교시]

In [1]:
import nltk
nltk.download('movie_reviews')
from nltk.corpus import movie_reviews

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\최지용\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


In [2]:
movie_reviews.fileids()[:10]  #문서아이디
movie_reviews.categories()
print( len(movie_reviews.fileids(categories='neg')) )
print( len(movie_reviews.fileids(categories='pos')) )

1000
1000


In [3]:
fieldid = movie_reviews.fileids()[0]
print(movie_reviews.raw(fieldid)[:100])
print( movie_reviews.categories(fieldid) )

plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 

['neg']


In [4]:
ids = movie_reviews.fileids()
reviews = [movie_reviews.raw(id) for id in ids]
categories = [movie_reviews.categories(id)[0]  for id in ids]

### TextBlob을 이용한 감성 분석

1: https://textblob.readthedocs.io/en/dev/

https://textblob.readthedocs.io/en/dev/quickstart.html

In [5]:
# 단어별 감성점수의 평균을 구하고, 강조어(very, extremely)나 부정어(not)에 대한 처리규칙이 포함
# %pip install textblob
!python -m textblob.download_corpora

Finished.


[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\최지용\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\최지용\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\최지용\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\최지용\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to
[nltk_data]     C:\Users\최지용\AppData\Roaming\nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\최지용\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-d

In [6]:
from textblob import TextBlob
result = TextBlob('this movie is very good')
result.sentiment
# polarity      -1 ~ 1   0 : 중립    1 : 매우 긍정
# subjectivity  0 ~ 1    0 : 객관적  1 : 매우 주관적

Sentiment(polarity=0.9099999999999999, subjectivity=0.7800000000000001)

In [7]:
from sklearn.metrics import accuracy_score
predict = [TextBlob(review).sentiment for review in reviews]
predict

[Sentiment(polarity=0.06479782948532947, subjectivity=0.5188408350908352),
 Sentiment(polarity=0.08063333333333332, subjectivity=0.33099999999999996),
 Sentiment(polarity=0.057845433255269293, subjectivity=0.5037861046057768),
 Sentiment(polarity=0.025467462644882, subjectivity=0.4826881720430107),
 Sentiment(polarity=0.022924603174603177, subjectivity=0.47592460317460306),
 Sentiment(polarity=0.04323376623376625, subjectivity=0.44196969696969696),
 Sentiment(polarity=0.0033339838667707585, subjectivity=0.5460902940411138),
 Sentiment(polarity=-0.05457711442786068, subjectivity=0.5166240227434257),
 Sentiment(polarity=0.030519769119769127, subjectivity=0.5424285714285714),
 Sentiment(polarity=-0.04998649105032083, subjectivity=0.46261524822695027),
 Sentiment(polarity=0.08487458619811562, subjectivity=0.4477075375604788),
 Sentiment(polarity=0.09467532467532466, subjectivity=0.5048809523809523),
 Sentiment(polarity=-0.048656060606060605, subjectivity=0.42670043290043297),
 Sentiment(po

In [8]:
import numpy as np
textblob_predict = [pred[0] for pred in predict]
textblob_predict = np.array(textblob_predict) > 0
textblob_predict = ['pos' if pred == True else 'neg' for pred in textblob_predict ]
accuracy_score(categories, textblob_predict)

0.6

In [9]:
# 특징 : 주관성 지수를 함께 제공
# 리뷰중에서 주관적 과 객관적의 비율

# [2교시]

### AFINN을 이용한 감성 분석

https://github.com/fnielsen/afinn 

(1) http://corpustext.com/reference/sentiment_afinn.html

In [10]:
# 감성점수를 -5 ~ +5 점수체계 : 처리속도가 빠름
%pip install afinn

Note: you may need to restart the kernel to use updated packages.


In [11]:
from afinn import Afinn

In [12]:
afn = Afinn(emoticons=True)
result = np.array([afn.score(review) for review in reviews]) > 0
result = ['pos' if re == True else 'neg' for re in result]
accuracy_score(categories, result)

0.664

### VADER를 이용한 감성 분석

(1) https://github.com/cjhutto/vaderSentiment

In [13]:
# 대문자 구두점 이모티콘 sns 데이터 특유의 감성표현을포착하는 '문법적 휴리스특'원리를 기술
%pip install vaderSentiment

Note: you may need to restart the kernel to use updated packages.


In [14]:
%pip install --upgrade vaderSentiment

Note: you may need to restart the kernel to use updated packages.


In [15]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm import tqdm
analyzer = SentimentIntensityAnalyzer()
result = []
for sentence in tqdm(reviews):
    vs = analyzer.polarity_scores(sentence)
    # print("{:-<65} {}".format(sentence[:10], str(vs)))
    if vs['compound'] > 0:
        result.append('pos')
    else:
        result.append('neg')

100%|██████████| 2000/2000 [00:49<00:00, 40.00it/s]


In [16]:
accuracy_score(categories,result)

0.639

## 2. 감성 사전 기반 분석 (Lexicon-based)

미리 정의된 감성 단어 사전을 사용하여 텍스트의 전체 감성 점수를 계산하는 방식입니다.

### 2.1 알고리즘 동작 원리 (소규모 데이터 증명)

**가정: 간단한 감성 사전**
- Positive: `great` (+2), `happy` (+3)
- Negative: `bad` (-2), `disappointing` (-3)

**시나리오 데이터**
- 문장 A: "The movie was **great** and I am **happy**."
- 문장 B: "The plot was **bad** and **disappointing**."

**증명 계산**
1. **문장 A 점수**: $Score(\text{"great"}) + Score(\text{"happy"}) = 2 + 3 = +5$ (긍정)
2. **문장 B 점수**: $Score(\text{"bad"}) + Score(\text{"disappointing"}) = (-2) + (-3) = -5$ (부정)

### 2.2 주요 감성 분석 도구별 상세 설명

사전 기반 분석을 지원하는 각 도구의 유래와 동작 특징은 다음과 같습니다.

#### 1) TextBlob
- **개요**: NLTK와 Pattern 라이브러리를 기반으로 하며, 직관적인 API를 제공하는 범용 NLP 라이브러리입니다.
- **특징**: 단순히 단어 점수를 더하는 것이 아니라, 문장 내 단어들의 상관관계(부정어, 강조어)를 파악하여 **평균 극성(Polarity)** 을 산출합니다. 또한 텍스트가 얼마나 주관적인지를 나타내는 **주관성(Subjectivity)** 지수를 함께 제공하는 것이 장점입니다.

#### 2) AFINN
- **개요**: Finn Årup Nielsen에 의해 구축된 가장 단순하고 직관적인 감성 사전 중 하나입니다.
- **특징**: 단어마다 -5(매우 부정)부터 +5(매우 긍정)까지의 **정수 점수** 가 매겨져 있습니다. 복잡한 문법 규칙 없이 텍스트에 포함된 단어 점수를 모두 더하는 방식이기 때문에 계산 속도가 매우 빠르고 성능이 안정적입니다.

#### 3) VADER (Valence Aware Dictionary and sEntiment Reasoner)
- **개요**: 소셜 미디어(SNS) 텍스트의 감성 분석을 위해 특별히 설계된 규칙 기반 모델입니다.
- **특징**: 감성 사전뿐만 아니라 **문법적 휴리스틱(Heuristics)** 을 사용합니다. 예를 들어, "GREAT"처럼 대문자로 쓰거나 "good!!!"처럼 느낌표를 사용하는 경우 감성 강도를 높게 평가합니다. 또한 "but"과 같은 접속사 뒤에 오는 문장에 더 높은 가중치를 두는 등 문맥 파악 능력이 뛰어납니다.

### 2.3 도구 간 알고리즘 및 활용 비교

| 특징 | **TextBlob** | **AFINN** | **VADER** |
| :--- | :--- | :--- | :--- |
| **기본 알고리즘** | 패턴 기반 (사전 + 규칙) | 단순 사전 합산 | 규칙 기반 (Social Media 특화) |
| **점수 산출 방식** | **평균 점수** (-1.0 ~ 1.0) | **총합 점수** (정수 위주) | **복합 점수** (Compound Score) |
| **부정어/강조어** | 지원 (매우 강력함) | 미지원 (단어별 독립) | 지원 (문맥 및 문장부호 포함) |
| **특이 사항** | 주관성(Subjectivity) 지수 제공 | 가장 빠르고 직관적임 | 대문자, 구두점, 이모티콘 반영 |
| **주요 활용** | 일반적인 문장 분석 | 빠른 키워드 기반 분석 | 소셜 미디어(SNS), 댓글 분석 |

#### 핵심 차이점 요약
1. **계산 방식**: TextBlob은 문장의 감성 단어 개수로 나누어 **밀도(평균)** 를 측정하는 반면, AFINN은 단순히 단어들의 가중치를 **누적**합니다.
2. **문맥 이해**: VADER는 "GOOD!!!"과 "good"의 차이를 인식하며, "but"과 같은 접속사가 감성 흐름을 바꾸는 것까지 고려하는 정교한 규칙을 가집니다.
3. **데이터 특성**: 분석하고자 하는 텍스트가 정제된 글이라면 TextBlob이 유리하고, 비격식적인 SNS 데이터라면 VADER가 가장 높은 성능을 보입니다.

### 2.4 TextBlob 알고리즘 상세 설명

TextBlob은 단순히 단어 점수를 더하는 것을 넘어, **패턴 기반 사후 처리(Pattern-based processing)** 를 수행합니다.

#### 동작 원리
1. **사전 매핑**: 문장 내 단어들을 사전에 정의된 **Polarity(극성)** 및 **Subjectivity(주관성)** 값과 매핑합니다.
2. **부정어 처리 (Negation)**: 'not', 'never' 등이 감성 단어 앞에 등장할 경우, 극성 값에 특정 가중치(보통 -0.5)를 곱하여 의미를 반전시킵니다.
3. **강조어 처리 (Intensity)**: 'very', 'really' 등은 뒤따르는 단어의 극성 절대값을 증폭시킵니다.
4. **최종 평균 산출**: 문장에서 발견된 모든 감성 단어의 수정된 점수들을 합산한 후, 단어 개수로 나누어 최종 극성값을 결정합니다.

# [3교시]

### 한국어 확장

1. **형태소 분석 및 원형 복원**: '좋아요', '좋다', '좋네요' 등을 하나의 감성 단어 '좋다'로 인식하기 위해 형태소 분석기(Okt 등)를 통한 어간 추출(Stemming)이 필수적입니다.
2. **한국어 전용 사전 구축**: '최고', '나쁘다' 등 한국어 감성어에 대한 극성값을 정의합니다.
3. **한국어 문법 규칙 적용**: '안 좋다', '좋지 않다'와 같이 한국어 특유의 부정 표현 방식을 반영하여 점수를 반전시키는 로직을 구현합니다.

In [17]:
import re
from konlpy.tag import Okt
class CustomTextBlob:
    '''TextBlob 모사
        1. 사전기반 점수 합산
        2. 부정어 처리
        3. 긍정어 처리
        4. 평균점수 산출
    '''
    def __init__(self, language='en'):
        self.language = language
        self.okt = Okt() if language == 'ko' else None
        # 감정어사전(원래는 수만개)
        self.lexicon = {
            'great' : 0.8, 'good' : 0.5, 'happy' : 0.7, 'bad' : -0.6, 'terrible' : -0.9, 'sad' : -0.5,
            '좋다' : 0.6, '최고' : 0.9, '행복하다' : 0.8, '나쁘다' : -0.6, '최악' : -0.9, '슬프다' : -0.7
        }
        # 부정어 / 강조어
        self.negation = {'not', 'never', 'no', '안', '못', '아니'}
        self.intersifiers = {
            'very' : 1.5, '매우' : 1.5
        }
    def tokenize(self, text):
        if self.language == 'ko':
            # 한국어는 형태소 분석 후 원형(stem) 추출
            return [word for word, pos in self.okt.pos(text, stem=True)]
        else:
            # 단순공백 및 특수문자 제거
            return re.findall(r'\w+', text.lower())      # [a-zA-Z0-9]
    def analyze(self, text):
        tokens = self.tokenize(text)
        scores = []
        negate = False
        intensity = 1.0
        for i, token in enumerate(tokens):
            # 강조어 확인
            if token in self.intersifiers:
                intensity = self.intersifiers[token]
                continue
            # 부정어 확인
            if token in self.negation:
                negate = True
                continue
            # 감정단어 점수 계산
            if token in self.lexicon:
                score = self.lexicon[token]*intensity
                # 부정어 처리
                if negate:
                    score *= -0.5
                    negate = False # 적용 후 해제
                scores.append(score)
                intensity = 1.0 # 적용 후 초기화

        if not scores:
            return 0.0
        # 평균 산출
        return sum(scores) / len(scores)

In [18]:
en_blob = CustomTextBlob()
en_blob.analyze('this movie is good')
re.findall(f'\w', 'this movie is good'.lower())

['t', 'h', 'i', 's', 'm', 'o', 'v', 'i', 'e', 'i', 's', 'g', 'o', 'o', 'd']

In [19]:
en_blob = CustomTextBlob(language='ko')
en_blob.analyze('이 영화는 최고로 좋다')
# re.findall(f'\w+', 'this movie is good')

0.75

In [20]:
a = [12, 2]
if not a:
    print('a')

### 1. 학습 목표
- Word2Vec의 두 가지 모델(CBOW, Skip-gram)의 작동 원리를 설명할 수 있다.
- 중심 단어와 주변 단어를 이용한 신경망 학습 과정을 이해한다.

### 2. 용어 정리
- **정의 (CBOW, Continuous Bag-of-Words):** 주변 단어(Context)들을 입력으로 받아 중심 단어(Target)를 예측하는 학습 방식.
- **사용 시나리오:** 전체적인 문맥이 주어졌을 때 빈칸에 들어갈 단어를 맞추는 문제와 유사.
- **구체적 예제:** [the, cat, (?), on, the] → (?) = "sits".

- **정의 (Skip-gram):** 중심 단어를 입력으로 받아 주변 단어들을 예측하는 학습 방식.
- **사용 시나리오:** 특정 단어가 주어졌을 때 그 주변에 어떤 단어들이 올지 추론할 때 활용.
- **구체적 예제:** [sits] → 주변 단어 {cat, on} 예측.

### 3. 이론적 배경
Word2Vec은 얕은 신경망(Input - Projection - Output) 구조를 가집니다. 학습이 끝나면 신경망의 가중치 행렬 $W$가 각 단어의 임베딩 벡터가 됩니다. 대규모 코퍼스를 통해 단어의 의미적 공간을 구축합니다.

### 4. 예제 코퍼스
- 문장: "I love cats"
- 윈도우 크기(Window Size): 1
- **Skip-gram 학습 데이터 쌍 (중심, 주변):** (love, I), (love, cats)

### 5. 수식 유도 및 직접 계산
**Softmax 기반 확률 계산:**
$$P(w_o | w_i) = \frac{\exp(\mathbf{v}_o^\top \mathbf{v}_i)}{\sum_{w=1}^V \exp(\mathbf{v}_w^\top \mathbf{v}_i)}$$

**계산 시나리오 (단어 $i$와 $o$의 유사도 점수가 $2$이고, 전체 점수 합이 $10$일 때):**
1. 분자: $\exp(2) \approx 7.39$
2. 분모(모든 단어의 지수 합): $10$ (가정)
3. 예측 확률: $P = 7.39 / 10 = 0.739$

기호 설명:
- $\mathbf{v}_i$: 입력 단어(중심 단어)의 벡터
- $\mathbf{v}_o$: 출력 단어(주변 단어)의 벡터
- $\mathbf{v}_w$: 전체 어휘 사전($V$) 내에 존재하는 임의의 단어 $w$의 벡터
- $V$: 전체 어휘 사전의 크기 (단어의 총 개수)

### 6. 비교 분석: CBOW vs Skip-gram
| 모델 | CBOW | Skip-gram |
| :--- | :--- | :--- |
| **예측 방향** | 주변 단어 → 중심 단어 | 중심 단어 → 주변 단어 |
| **학습 속도** | 빠름 | 상대적으로 느림 |
| **성능 특징** | 흔한 단어 예측에 강함 | 희귀 단어 및 대량 데이터에 강함 (일반적 권장) |

### 실습 Gensim 라이브러리

In [21]:
%pip install gensim

Note: you may need to restart the kernel to use updated packages.


In [22]:
from gensim.models import Word2Vec

In [23]:
corpus = [
    'i love nlp',
    'i love machine learning',
    'nlp is fun^^',
    'machine learning is powerfull',
    'i enjoy deep learning',
    'natural language processing is interesting'
]
import re
sentences = [re.findall(r'\w+',text.lower()) for text in corpus]
# word2vec 모델 학습
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=3,       # 주변 단어 범위
    min_count = 1,  # 최소 등장 횟수
    sg = 1 # 0 : CBOW 1=skip-gram
)
# 단어 벡터 확인
vector = model.wv['love']
# 유사 단어 찾기
model.wv.most_similar('learning')

[('fun', 0.13724106550216675),
 ('machine', 0.06804422289133072),
 ('language', 0.0338103361427784),
 ('love', 0.009400710463523865),
 ('enjoy', 0.008315923623740673),
 ('nlp', 0.00449696509167552),
 ('powerfull', -0.003653511870652437),
 ('is', -0.010894806124269962),
 ('i', -0.023647421970963478),
 ('natural', -0.09577618539333344)]

In [24]:
# 유사도 계산
model.wv.similarity('fun', 'learning')

np.float32(0.13724104)

In [25]:
model.wv.index_to_key
model.save('customword2vec.model')
loaded_model = Word2Vec.load('customword2vec.model')

In [26]:
# 한국어 지원
import pandas as pd
df = pd.read_csv('daum_movie_review.csv')
corpus =  df['review'][:100]
corpus  = corpus.to_numpy()

# [4교시]

In [27]:
import re
from konlpy.tag import Okt
okt = Okt()
# 전처리
sentences = [re.sub(r'[^가-힣\s]','',text) for text in corpus]
# 형태소분리
def korean_token(text):
    return [word for word,_  in okt.pos(text,stem=True) if len(word)> 1]
    

sentences = [korean_token(doc) for doc in sentences]
# 나머지는 동일하게 적용
# word2vec 모델 학습
model = Word2Vec(
    sentences=sentences,
    vector_size=1000,
    window=3,  # 주변단어 범위
    min_count=5 ,  #최소등장횟수
    sg = 1  # 0: CBOW  1=skip-gram
)
# 유사단어 찾기
model.wv.most_similar('영화')

[('가다', 0.10001443326473236),
 ('있다', 0.08331053704023361),
 ('다음', 0.073957659304142),
 ('같다', 0.06599206477403641),
 ('모르다', 0.05515427514910698),
 ('자다', 0.054868899285793304),
 ('나오다', 0.05061757564544678),
 ('그래도', 0.04825276881456375),
 ('재밌다', 0.0442919097840786),
 ('정말', 0.025044571608304977)]

In [28]:
# 다음리뷰데이터 로드
# 평점 최저 최고 점수의 절반을 threshold로 정하고 0이면 긍정, 1이면 부정으로 추가 컬럼 생성

In [29]:
import pandas as pd
df = pd.read_csv('daum_movie_review.csv')

In [30]:
df['rating'].max(), df['rating'].min()

(10, 0)

In [31]:
df['target'] = df['rating'].apply(lambda x : 0 if x > 5 else 1)

In [32]:
# 전처리
import re
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
df['cleaned_review'] = df['review'].apply(lambda x : re.sub(r'[^가-힣\s]','',x) )
# 토큰화(한글은 형태소)
okt = Okt()
def kor_tokenize(text):
    return [word for word, _ in okt.pos(text, stem=True)]

tfidf = TfidfVectorizer(tokenizer=kor_tokenize,max_features=5000)
x_tfidf = tfidf.fit_transform(df['cleaned_review']).toarray()
y = df['target'].values

c:\miniconda\envs\ml_env\lib\site-packages\sklearn\feature_extraction\text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


# [5교시]

In [33]:
from torch.utils.data import Dataset, DataLoader
import torch
class TfidfDataset(Dataset):
    def __init__(self, vectors, labels):
        self.vectors = torch.FloatTensor(vectors)
        self.labels = torch.FloatTensor(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return self.vectors[index], self.labels[index]

In [34]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x_tfidf, y, random_state=42, test_size=0.2)

train_loader = DataLoader(TfidfDataset(x_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TfidfDataset(x_test, y_test), batch_size=32, shuffle=True)


In [35]:
import torch.nn as nn
class TfidfMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim,hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim,1),
        )
    def forward(self, x):
        return self.network(x)    

In [36]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [37]:
from torch.optim import Adam
x_train.shape[1]
model = TfidfMLP(input_dim = x_train.shape[1], hidden_dim = 64).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), lr=1e-3)

In [38]:
from tqdm import tqdm
epochs = 10

for epoch in tqdm(range(epochs)):
    model.train()
    local_loss = 0.0
    for vecs,labels in train_loader:
        vecs, labels = vecs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(vecs).squeeze(1),labels)
        loss.backward()
        optimizer.step()
        local_loss += loss.item()
    print(f'epoch : {epoch+1}/{epochs} loss : {(local_loss / len(train_loader)):.4f}')

 10%|█         | 1/10 [00:01<00:15,  1.76s/it]

epoch : 1/10 loss : 0.5109


 20%|██        | 2/10 [00:03<00:13,  1.72s/it]

epoch : 2/10 loss : 0.3339


 30%|███       | 3/10 [00:05<00:12,  1.71s/it]

...

epoch : 8/10 loss : 0.1574


 90%|█████████ | 9/10 [00:16<00:01,  1.86s/it]

epoch : 9/10 loss : 0.1419


100%|██████████| 10/10 [00:18<00:00,  1.86s/it]

epoch : 10/10 loss : 0.1301


In [39]:
# 예측... 정확도
model.eval()
correct  = 0.0; total = 0.0
with torch.no_grad():
    for vecs, labels in test_loader:
        vecs,labels = vecs.to(device), labels.to(device)
        preds = (torch.sigmoid(model(vecs)).squeeze(1) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.shape[0]
    print(f'test accuracy : {(correct / total):.4f}')

test accuracy : 0.8407


# [6교시] ~ [8교시]

## 신경망_embedding

In [40]:
import pandas as pd
df = pd.read_csv('daum_movie_review.csv')
df.head()

,review,rating,date,title
0,돈 들인건 티가 나지만 보는 내내 하품만,1,2018.10.29,인피니티 워
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,2018.10.26,인피니티 워
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,2018.10.24,인피니티 워
3,이 정도면 볼만하다고 할 수 있음!,8,2018.10.22,인피니티 워
4,재미있다,10,2018.10.20,인피니티 워


In [41]:
# 전처리
import re
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
df['cleaned_review'] = df['review'].apply(lambda x : re.sub(r'[^가-힣\s]','',x) )
# 토큰화(한글은 형태소)
okt = Okt()
def kor_tokenize(text):
    return [word for word, pos in okt.pos(text, stem=True) if pos in ['Noun','Verb','Adjective']]

df['data'] = df['cleaned_review'].apply(kor_tokenize)
df.head()

,review,rating,date,title,cleaned_review,data
0,돈 들인건 티가 나지만 보는 내내 하품만,1,2018.10.29,인피니티 워,돈 들인건 티가 나지만 보는 내내 하품만,"[돈, 들이다, 티, 나, 보다, 내내, 하품]"
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,2018.10.26,인피니티 워,몰입할수밖에 없다 어렵게 생각할 필요없다 내가 전투에 참여한듯 손에 땀이남,"[몰입, 하다, 없다, 어렵다, 생각, 하다, 필요없다, 내, 전투, 참여, 듯, ..."
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,2018.10.24,인피니티 워,이전 작품에 비해 더 화려하고 스케일도 커졌지만 전국 맛집의 음식들을 한데 모은 것...,"[이전, 작품, 비다, 더, 화려하다, 스케일, 커지다, 전국, 맛집, 음식, 모으..."
3,이 정도면 볼만하다고 할 수 있음!,8,2018.10.22,인피니티 워,이 정도면 볼만하다고 할 수 있음,"[이, 정도, 볼, 하다, 하다, 수, 있다]"
4,재미있다,10,2018.10.20,인피니티 워,재미있다,[재미있다]


In [42]:
# 단어사전 구축
from collections import Counter
all_tokens = [token for sublist in df['data'].tolist() for token in sublist]
vocab_counts = Counter(all_tokens)

vocab = { word : 0  for word,count in vocab_counts.items() if count >=2}
for i,voc in enumerate(vocab):
    vocab[voc] = i+2

vocab['<PAD>'] = 0
vocab['<UNK>'] = 1
vocab_size = len(vocab)
print(vocab_size)
def text_to_sequence(tokens, vocab):
    return [vocab.get(token, vocab['<UNK>']) for token in tokens]

df['sequence'] = df['data'].apply(lambda x : text_to_sequence(x, vocab))

6774


In [43]:
df.head()

,review,rating,date,title,cleaned_review,data,sequence
0,돈 들인건 티가 나지만 보는 내내 하품만,1,2018.10.29,인피니티 워,돈 들인건 티가 나지만 보는 내내 하품만,"[돈, 들이다, 티, 나, 보다, 내내, 하품]","[2, 3, 4, 5, 6, 7, 8]"
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,2018.10.26,인피니티 워,몰입할수밖에 없다 어렵게 생각할 필요없다 내가 전투에 참여한듯 손에 땀이남,"[몰입, 하다, 없다, 어렵다, 생각, 하다, 필요없다, 내, 전투, 참여, 듯, ...","[9, 10, 11, 12, 13, 10, 14, 15, 16, 17, 18, 19..."
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,2018.10.24,인피니티 워,이전 작품에 비해 더 화려하고 스케일도 커졌지만 전국 맛집의 음식들을 한데 모은 것...,"[이전, 작품, 비다, 더, 화려하다, 스케일, 커지다, 전국, 맛집, 음식, 모으...","[22, 23, 24, 25, 26, 27, 28, 29, 1, 30, 31, 32..."
3,이 정도면 볼만하다고 할 수 있음!,8,2018.10.22,인피니티 워,이 정도면 볼만하다고 할 수 있음,"[이, 정도, 볼, 하다, 하다, 수, 있다]","[43, 44, 45, 10, 10, 46, 47]"
4,재미있다,10,2018.10.20,인피니티 워,재미있다,[재미있다],[48]


In [44]:
MAX_LEN = 50
def pad_sequence(seq, max_len):
    if len(seq) < max_len:
        return seq + [0]*(max_len - len(seq))
    else:
        return seq[:max_len]

df['padded']  = df['sequence'].apply(lambda x : pad_sequence(x,MAX_LEN))

In [45]:
len(df['padded'][100])

50

In [46]:
from torch.utils.data import Dataset, DataLoader
import torch
class MovieReviewerDataset(Dataset):
    def __init__(self,sequences, labels):
        self.sequences = torch.LongTensor(sequences)
        self.labels = torch.FloatTensor(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return self.sequences[index], self.labels[index]

In [47]:
from sklearn.model_selection import train_test_split
df['target'] =  df['rating'].apply(lambda x : 0 if x > 5 else 1)

x_train,x_test,y_train,y_test = train_test_split(df['padded'].tolist(),df['target'].values,random_state=42,test_size=0.2)
train_loader = DataLoader(MovieReviewerDataset(x_train,y_train), batch_size=32,shuffle=True)
test_loader = DataLoader(MovieReviewerDataset(x_test, y_test), batch_size=32, shuffle=False)

# [7교시]

In [48]:
import torch.nn as nn
class EmbeddingMLP(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim,padding_idx=0)
        self.fc = nn.Sequential(            
            nn.Linear(embedding_dim,hidden_dim),
            nn.Dropout(0.5),
            nn.ReLU(),
            nn.Linear(hidden_dim,1)
        )
    def forward(self, x):  
        # B L D --> B D   배치사이즈, 시퀀스 길이, 임베딩차수
        x = self.embedding(x)      
        x = x.mean(dim=1)
        return self.fc(x)

In [52]:
# 임베딩 검증
import numpy as np
vec, label = next(iter(train_loader))
temp = vec.reshape(-1)
temp = nn.Embedding(vocab_size,128,padding_idx=0)(vec)
temp[0], temp.shape
# vec
# vocab_size,temp.max()

(tensor([[-0.4511,  0.2192,  0.5973,  ...,  0.8818,  1.0205,  1.5659],
         [-0.8381, -0.6683,  0.8916,  ..., -0.7726, -0.7714, -0.0410],
         [-0.7004, -0.5745, -0.4559,  ..., -2.0149,  2.1557,  1.8178],
         ...,
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
        grad_fn=<SelectBackward0>),
 torch.Size([32, 50, 128]))

In [50]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
from torch.optim import Adam
model = EmbeddingMLP(vocab_size, np.array(x_train).shape[1], 64).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), lr=1e-3)

from tqdm import tqdm
epochs = 10
pbar = tqdm(range(epochs), desc="Training")
for epoch in pbar:
    model.train()
    local_loss = 0.0
    for vecs, labels in train_loader:
        vecs, labels = vecs.to(device), labels.to(device)
        optimizer.zero_grad()        
        outputs = model(vecs).squeeze(1)        
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        local_loss += loss.item()
    avg_loss = local_loss / len(train_loader)
    pbar.set_postfix(
        loss=f"{avg_loss:.4f}"
    )   

Training: 100%|██████████| 10/10 [00:07<00:00,  1.27it/s, loss=0.2002]


# [8교시]

In [51]:
# 예측... 정확도
model.eval()
correct  = 0.0; total = 0.0
with torch.no_grad():
    for vecs, labels in test_loader:
        vecs,labels = vecs.to(device), labels.to(device)
        preds = (torch.sigmoid(model(vecs)).squeeze(1) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.shape[0]
    print(f'test accuracy : {(correct / total):.4f}')

test accuracy : 0.8445


### 예시
나는 고양이를 키운다.
나는 오늘 고기를 먹었다.
---
나는 고양이 키운다
나는 오늘 고기 를 먹었 다
---
나는:2 고양이:1 키운다:1 오늘:2 고기:1 를:1 먹었:1 다:1

0~7

나는: 2
오늘: 5

In [53]:
# 문장의 길이를 히스토그램으로 시각화해서 가장 적당한 길이를 찾는다
# 최대한 많은 정보를 가지는 길이를 어느정도 유추